# Notebook showing pipeline usage. 

In this notebook, we show how to use the `Pipeline` class for running and end-to-end quantum kernel model.

## 1. Imports 

In [9]:
import pandas as pd 

import qkernel

from qkernel4eo.embeddings import ChainEmbedding, RadialEmbedding

## 2. Dataset loading


In [10]:
df_train = pd.read_csv(
    "./train_toy.csv"
).rename(columns=str.lower)
df_test = pd.read_csv(
    "./test_toy.csv"
).rename(columns=str.lower)

In [7]:
x_train = df_train[["b03", "b04", "b08"]].to_numpy()
x_test = df_test[["b03", "b04", "b08"]].to_numpy()
y_train = df_train["label"].to_numpy()
y_test = df_test["label"].to_numpy()

## 3. Different embeddings

We implement different embeddings for our EO data, from the [qkernel4eo package](https://github.com/ICHEC/qkernel4eo).

In [11]:
x_train_chain = ChainEmbedding(x_train).embed()
x_test_chain = ChainEmbedding(x_test).embed()

x_train_radial = RadialEmbedding(x_train).embed()
x_test_radial = RadialEmbedding(x_test).embed()



## 4. Implementing the pipeline

`pipeline.py` file takes as arguments the coordinates `x_train`, `x_test`, and the labels `y_train`, `y_test`. It also takes in a dictionary `args` with the optional arguments for our simulations. The dictionary keys will be the following: 


### 4.1.Hamiltonian.
Dictionary containing hamiltonian options:
```ini
- "backend": str
    Backend to use in the hamiltonian. It must be one of 
    ["myqlm", "qutip"].
- "kwargs": dict
    Optional keyboard args for hamiltonian construction. 
    These are :
    - "omega": float
        Rabi frequency (default: 2*np.pi)
    - "delta": float 
        Detuning (default: 0)
    - "c6": float
        Interaction strength (default: 865723.02)
```

Example of how this dictionary would look like:
```ini
"hamiltonian": {
    "backend": "myqlm",
    "kwargs": {
        "omega": 2 * np.pi,
        "delta": 0.3,
        "c6": 865723.02
    }
}
```

### 4.2.Simulator. 
Dictionary containing simulator options:
```ini
- "backend": str
    Backend to use. It must be one of 
    ["myqlm", "qutip"]. It must be the same as the one used 
    in the hamiltonian. 
- "kwargs": dict
    Optional keyword arguments for the simulation. These are:
    - "duration": float
        Duration of evolution time for the simulation. (default: 0.66)
```

Example of how this dictionary would look like:
```ini
"simulator": {
    "backend": "myqlm",
    "kwargs": {
        "duration": 0.68
    }
}
```

### 4.3.Kernel. 
Dictionary containing options for kernel computation:
```ini
- "kwargs": dict
    Optional keyword arguments for kernel computation. These are:
    - "excitations": bool
        Whether to use excitations or basis state as observables. (default: True)
    - "distance_fn": str
        Distance to use to compute kernel. Must be one of ["exp_js", "exp"], 
        corresponding to exponential jensen shannon and exponential distances
        that can be found on the file `kernel.py`. (default : exp_js)
    - "distance_kwargs": dict
        Optional keyword arguments for computing the distance. These are:
        - "mu": float
            Scaling term for the exponential. 
```

Example of how this dictionary would look like:
```ini
"kernel": {
    "kwargs": {
        "excitations": False,
        "distance_fn": "exp",
        "distance_kwargs": {"mu": 0.75},
    }
}
```



### 4.4.Model.
Dictionary containing options for model computation:
```ini
- "type": str 
    Type of kernel model to use. Must be one of ["svc"]
- "kwargs": dict
    Optional keyword arguments for the kernel model. Those 
    arguments will vary depending on the selected model and 
    can be found in model.py file.
```

Example of how this dictionary would look like:
```ini
"model": {
    "type": "svc",
    "kwargs": {
        "C": 0.9,
        "tol": 0.02}
}
```

### 4.5. Results path
Path and name where to save results. Example:
```ini
"results": {
    "path" :  "./../../results/experiment_trial",
    "name" : "experiment_trial"
}
```

Here is an example of how the args dictionary would look like:

In [ ]:
import numpy as np 

pipeline_args = {"hamiltonian": {
        "backend": "qutip",
        "kwargs": {
            "omega": 2 * np.pi,
            "delta": 0.3,
            "c6": 865723.02
        }
    },
    "simulator": {
        "backend": "qutip",
        "kwargs": {
            "duration": 0.68
        }
    },
    "kernel": {
        "kwargs": {
            "excitations": False,
            "distance_fn": "exp",
            "distance_kwargs": {
                "mu": 0.75
            },
        }
    },
    "model": {
        "type": "svc",
        "kwargs": {
            "C": 0.9,
            "tol": 0.02
        }
    },
    "results": {
    "path" :  "./",
    "name" : "experiment_trial"
    }
}



And here we run the pipeline for the two defined embeddings:

In [20]:
pipeline_chain = qkernel.pipeline.Pipeline(
    x_train_chain, x_test_chain, y_train, y_test, pipeline_args
)
pipeline_chain.run()

pipeline_radial = qkernel.pipeline.Pipeline(
    x_train_radial, x_test_radial, y_train, y_test, pipeline_args
)
pipeline_radial.run()